In [1]:
import os

import bids
from bids import BIDSLayout
import nibabel as nb
import warnings

import pandas as pd
import numpy as np
import json
import gzip

from nilearn import image as nimg
from nilearn import plotting as nplot

import seaborn as sns
import ptitprince as pt
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

from scipy import stats

sns.set_theme(style='whitegrid', palette='bright', color_codes=True)
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
data_path = "openpain.org/subacute_longitudinal_study/"
layout = BIDSLayout(data_path)

In [3]:
layout.get_sessions()

['visit1', 'visit4', 'visit2', 'visit3', 'visit5']

In [4]:
layout_df = layout.to_df()
layout_df.to_excel("layout.xlsx")

### 2.1 Checking the number of visits present for each participant

In [5]:
def get_number_of_sessions_for_each_subject():
    """
    Returns a dict that contains the number of sessions a participant was present for.
    Here the key = participant_id, and the value = count os sessions.
    """
    participant_and_sessions = {}
    parent_dir = "openpain.org/subacute_longitudinal_study/"
    for folder in os.listdir(parent_dir):
        if folder.startswith("sub"):
            subject_dir = os.path.join(parent_dir, folder)
            count = 0
            for subject_folder in os.listdir(subject_dir):
                if subject_folder.startswith("ses"):
                    count += 1
            participant_and_sessions[subject_dir[-3:]] = count
    
    participant_and_sessions = dict(sorted(participant_and_sessions.items(), key=lambda kv: kv[0]))
    return participant_and_sessions

In [6]:
participants_and_sessions = get_number_of_sessions_for_each_subject()

In [7]:
count_4, count_5 = 0, 0
for key, value in participants_and_sessions.items():
    if value == 4:
        count_4 += 1
    if value == 5:
        count_5 += 1
print(count_4, count_5)

82 40


### 2.2 Checking the number of files present for each visit of each participant

In [8]:
def get_number_of_files_for_each_session_for_each_subject():
    """
    Returns a dict that contains the number of files for each session a participant was present for.
    """
    participant_and_files = {}
    parent_dir = "openpain.org/subacute_longitudinal_study/"
    for folder in sorted(os.listdir(parent_dir)):
        if folder.startswith("sub"):
            subject_dir = os.path.join(parent_dir, folder)
            participant_and_files[folder] = {}
            for session_folder in sorted(os.listdir(subject_dir)):
                if session_folder.startswith("ses"):
                    data_dir = os.path.join(subject_dir, session_folder)
                    participant_and_files[folder][session_folder] = {}
                    for data_folder in sorted(os.listdir(data_dir)):
                        final_folder = os.path.join(data_dir, data_folder)
                        count = 0
                        try:
                            for files in sorted(os.listdir(final_folder)):
                                count += 1
                            participant_and_files[folder][session_folder][data_folder] = count
                        except:
                            print(final_folder, "is not a folder")
    
    return participant_and_files

In [9]:
participant_and_files = get_number_of_files_for_each_session_for_each_subject()

openpain.org/subacute_longitudinal_study/sub-014/ses-visit1/.DS_Store is not a folder
openpain.org/subacute_longitudinal_study/sub-014/ses-visit3/.DS_Store is not a folder


In [10]:
print(json.dumps(participant_and_files, indent=2))

{
  "sub-001": {
    "ses-visit1": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit2": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit3": {
      "anat": 0,
      "dwi": 0,
      "func": 0
    },
    "ses-visit4": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit5": {
      "anat": 1,
      "dwi": 3,
      "func": 3
    }
  },
  "sub-002": {
    "ses-visit1": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit2": {
      "anat": 1,
      "dwi": 3,
      "func": 2
    },
    "ses-visit3": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit4": {
      "anat": 1,
      "dwi": 3,
      "func": 7
    }
  },
  "sub-003": {
    "ses-visit1": {
      "anat": 1,
      "dwi": 3,
      "func": 5
    },
    "ses-visit2": {
      "anat": 0,
      "dwi": 3,
      "func": 9
    },
    "ses-visit3": {
      "anat": 1,
      "dwi": 3,
      "func": 9
    },
    "ses-visit4": {
      "anat":

### 2.3 Counting valid (has 9 or more files in func folder) participant sessions

In [11]:
def get_valid_participant_sessions():
    valid_participant_sessions = {}
    for subject in participant_and_files:
        for session in participant_and_files[subject]:
            for folder in participant_and_files[subject][session]:
                if folder == "func":
                    if participant_and_files[subject][session][folder] >= 9:
                        valid_participant_sessions[subject + "_" + session + "_" + folder] = "valid"
                    else:
                        valid_participant_sessions[subject + "_" + session + "_" + folder] = "invalid"
    return valid_participant_sessions

In [12]:
valid_participant_sessions = get_valid_participant_sessions()

In [13]:
count_valid, count_invalid = 0, 0
for key, value in valid_participant_sessions.items():
    if value == "valid":
        count_valid += 1
    if value == "invalid":
        count_invalid += 1
print(count_valid, count_invalid)

208 320


sub-035 has a 10th file called: sub-035_ses-visit4_task-resting_bold.nii.gz therefore condition for valid is 9 files or greater.

In [14]:
print(valid_participant_sessions)

{'sub-001_ses-visit1_func': 'valid', 'sub-001_ses-visit2_func': 'valid', 'sub-001_ses-visit3_func': 'invalid', 'sub-001_ses-visit4_func': 'valid', 'sub-001_ses-visit5_func': 'invalid', 'sub-002_ses-visit1_func': 'valid', 'sub-002_ses-visit2_func': 'invalid', 'sub-002_ses-visit3_func': 'valid', 'sub-002_ses-visit4_func': 'invalid', 'sub-003_ses-visit1_func': 'invalid', 'sub-003_ses-visit2_func': 'valid', 'sub-003_ses-visit3_func': 'valid', 'sub-003_ses-visit4_func': 'invalid', 'sub-003_ses-visit5_func': 'invalid', 'sub-004_ses-visit1_func': 'valid', 'sub-004_ses-visit2_func': 'valid', 'sub-004_ses-visit3_func': 'valid', 'sub-004_ses-visit4_func': 'invalid', 'sub-005_ses-visit1_func': 'invalid', 'sub-005_ses-visit2_func': 'invalid', 'sub-005_ses-visit3_func': 'valid', 'sub-005_ses-visit4_func': 'invalid', 'sub-006_ses-visit1_func': 'valid', 'sub-006_ses-visit2_func': 'invalid', 'sub-006_ses-visit3_func': 'valid', 'sub-006_ses-visit4_func': 'valid', 'sub-006_ses-visit5_func': 'invalid', '